# Adım 5 — Gerçek eğitim + ölçüm (BERT / RoBERTa, leave-one-dataset-out)

**Amaç:** `veri_birlesik.csv` ile BERT ve RoBERTa'yı eğitip, **eğitimde hiç görmediği** veri setlerinde ölçmek (history.md, Karar 8).

**Nasıl çalıştırılır:**
1. Üst menüden `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`.
2. Hücreleri **sırayla** çalıştır. Önce **"TEK KOŞU"** hücresine kadar git. Bu hücre sadece 1 koşu yapar (safeguard'ı dışarıda bırak, BERT). Çıktısını Claude'a yapıştır.
3. Tek koşu düzgün çıkarsa "TÜM KOŞULAR" hücresini çalıştır. Hata çıkarsa hatayı olduğu gibi Claude'a yapıştır.

**Önemli:** Bu defter henüz gerçek veriyle ve GPU ile **denenmedi**. Mantık sahte veriyle ve minik bir modelle test edildi.

## 1) Kütüphaneler

In [ ]:
!pip -q install transformers scikit-learn

## 2) Gerekli parçalar ve ayarlar
`TOHUM` her koşuda aynı rastgelelik için sabit (02 ile aynı: 42). Aşağıdaki ayar değerleri Karar 8'den geliyor.

In [ ]:
import os, time, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GroupShuffleSplit

# ---- Ayarlar (Karar 8) ----
TOHUM = 42            # her koşuda aynı rastgelelik
TUR = 3               # veriyi kaç kez baştan sona gezecek
PARTI = 16            # aynı anda kaç örnek işlenecek
OGRENME_HIZI = 2e-5   # her adımda ne kadar düzeltme (küçük tutulur)
MAKS_UZUNLUK = 128    # uzun metinler bu kadar parçada kesilir (bildiride sınırlılık olarak yazılacak)
HEDEF_FPR = 0.01      # %1 yanlış pozitif oranı

cihaz = "cuda" if torch.cuda.is_available() else "cpu"
KARISIK = (cihaz == "cuda")   # GPU'da hızlandırmak için yarı hassasiyet (fp16)
print("Kullanılan cihaz:", cihaz)
if cihaz == "cpu":
    print("UYARI: GPU yok! Çalışma zamanı türünden T4 GPU seçmelisin, yoksa eğitim çok yavaş olur.")
else:
    print("Kart:", torch.cuda.get_device_name(0))


def tohum_ayarla(t=TOHUM):
    random.seed(t); np.random.seed(t); torch.manual_seed(t)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(t)

## 3) Google Drive'ı bağla (sonuçları kaybetmemek için)
Sonuç tablosu her koşudan sonra **senin** Drive'ındaki `YiitliProje_sonuclar` klasörüne yazılır. Colab kopsa bile kaybolmaz. Bağlanırken izin penceresi çıkacak, izin ver.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SONUC_KLASORU = "/content/drive/MyDrive/YiitliProje_sonuclar"
os.makedirs(SONUC_KLASORU, exist_ok=True)
SONUC_DOSYASI = SONUC_KLASORU + "/sonuclar_ahmet.csv"   # her koşunun ölçümleri
HATA_DOSYASI = SONUC_KLASORU + "/hatalar_ahmet.csv"     # modelin yanıldığı örnekler
print("Sonuçlar şuraya yazılacak:", SONUC_DOSYASI)

## 4) veri_birlesik.csv'yi indir ve kontrol et
Dosya Yiğit'in Drive'ında. Aşağıdaki kod Google hesabınla Drive'dan dosyayı **dosya numarasıyla** indirir (Yiğit seni klasöre eklediği için erişimin var). Kontrol: **23.081 satır x 7 sütun** çıkmalı.

In [ ]:
from google.colab import auth
auth.authenticate_user()                       # Google hesabınla giriş
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

DOSYA_ID = "BURAYA_DRIVE_DOSYA_NUMARASI"   # veri_birlesik.csv'nin Drive numarası
servis = build("drive", "v3")
istek = servis.files().get_media(fileId=DOSYA_ID)
with open("veri_birlesik.csv", "wb") as f:
    indirici = MediaIoBaseDownload(f, istek)
    bitti = False
    while not bitti:
        _, bitti = indirici.next_chunk()
print("İndirildi. Boyut (bayt):", os.path.getsize("veri_birlesik.csv"))

In [ ]:
df = pd.read_csv("veri_birlesik.csv")
print("Şekil:", df.shape, "(beklenen: 23081 x 7)")
print("Sütunlar:", list(df.columns))
print()
print(df.groupby(["rol", "kaynak", "bolum"]).agg(satir=("label", "size"), zararli=("label", "sum")))

# Basit güvenlik kontrolleri: beklediğimiz değerler yoksa burada dururuz
beklenen = {"safeguard", "neuralchemy", "wambosec", "jackhhao", "deepset"}
assert beklenen <= set(df["kaynak"]), f"Beklenen kaynak adları yok! Bulunan: {set(df['kaynak'])}"
assert set(df["bolum"]) <= {"train", "val", "test"}, f"Beklenmeyen bölüm adı: {set(df['bolum'])}"
assert set(df["label"]) <= {0, 1}, "label sadece 0 ve 1 olmalı"
print("\nBoş metin sayısı:", df["text"].isna().sum())
df = df.dropna(subset=["text"]).reset_index(drop=True)
df["text"] = df["text"].astype(str)

## 5) Veriyi koşuya göre bölen fonksiyon (Karar 8)
- **Eğitim havuzu** = çıkarılmayan 2 ana veri setinin `train` + `val` bölümleri.
- Havuzun **%10'u** "eşik seçme seti" olarak ayrılır (modele gösterilmez). Sadece "%1 yanlış pozitifte hangi eşik?" sorusu için kullanılır.
- **Test** = çıkarılan veri setinin **tamamı** (model hiç görmedi) + `jackhhao (test)` + `deepset (tamamı)`.
- `kaynak_cikarilan=None` verilirse **aynı dağılımda referans** koşusu: 3 ana veri setiyle eğit, 3'ünün `test` bölümünde ölç (genelleme sayılmaz, sadece kıyas için).

neuralchemy'de aynı metnin çoğaltılmış hâlleri `grup` sütunuyla işaretli. %10'luk ayırmayı **gruba göre** yapıyoruz ki aynı grubun parçaları hem eğitime hem eşik setine düşüp eşiği olduğundan iyi göstermesin.

In [ ]:
def egitim_ayir(havuz):
    """Havuzu %90 eğitim / %10 eşik-seçme setine böler (grup bütünlüğünü korur)."""
    havuz = havuz.reset_index(drop=True)
    # grup boşsa her satır kendi başına bir grup sayılır
    gruplar = [f"tek_{i}" if pd.isna(g) else str(g) for i, g in enumerate(havuz["grup"])]
    ayirici = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=TOHUM)
    egitim_idx, esik_idx = next(ayirici.split(havuz, groups=gruplar))
    return havuz.iloc[egitim_idx].reset_index(drop=True), havuz.iloc[esik_idx].reset_index(drop=True)


def kosu_verisi(df, kaynak_cikarilan):
    ana = df[df["rol"] == "ana"]
    dis = df[df["rol"] == "dis_test"]
    if kaynak_cikarilan is None:
        havuz = ana[ana["bolum"].isin(["train", "val"])]
        testler = {}
        for k in sorted(ana["kaynak"].unique()):
            testler[f"{k} (test)"] = ana[(ana["kaynak"] == k) & (ana["bolum"] == "test")]
        testler["ana_uc_test (birlesik)"] = ana[ana["bolum"] == "test"]
    else:
        havuz = ana[(ana["kaynak"] != kaynak_cikarilan) & (ana["bolum"].isin(["train", "val"]))]
        testler = {f"{kaynak_cikarilan} (tamami)": ana[ana["kaynak"] == kaynak_cikarilan]}
    testler["jackhhao (test)"] = dis[(dis["kaynak"] == "jackhhao") & (dis["bolum"] == "test")]
    testler["deepset (tamami)"] = dis[dis["kaynak"] == "deepset"]
    egitim, esik_seti = egitim_ayir(havuz)
    return egitim, esik_seti, testler

## 6) Ölçüt fonksiyonları
Model her metin için "zararlı olma olasılığı" (0 ile 1 arası) üretir. Karar vermek için bir **eşik** gerekir: olasılık eşikten büyükse "zararlı".

- **Varsayılan eşik 0,5:** accuracy, precision, recall, F1 bununla hesaplanır.
- **ROC-AUC:** Eşikten bağımsız genel ayırma gücü (1,0 mükemmel, 0,5 şans).
- **TPR@%1 FPR (a), iyimser:** Test setinin kendi ROC eğrisinde, masumların en fazla %1'ini yanlış engelleyen en iyi eşikte yakalama oranı. Eşiği test setine bakarak seçtiği için iyimserdir.
- **TPR@%1 FPR (b), gerçekçi:** Eşiği **eşik seçme setinde** (modelin görmediği, eğitim dağılımından) seçip **sabit** tutarız. Görülmemiş testte gerçek yakalama oranını (TPR) ve gerçek yanlış pozitif oranını (FPR) raporlarız. Yeni dağılımda FPR %1'den büyük çıkabilir, bu beklenen ve önemli bir bulgudur.

In [ ]:
def esik_sec(y_esik, p_esik, hedef_fpr=HEDEF_FPR):
    """Masumların (label=0) en fazla hedef_fpr kadarı eşiğin üstünde kalacak şekilde eşik seç.
    Karar kuralı: olasılık > eşik ise 'zararlı'."""
    masum = np.sort(p_esik[y_esik == 0])
    k = int(np.floor(hedef_fpr * len(masum)))   # izin verilen yanlış engelleme sayısı
    return float(masum[len(masum) - 1 - k])


def tpr_roc(y, p, hedef_fpr=HEDEF_FPR):
    """Test setinin kendi ROC eğrisinden: FPR <= hedef iken ulaşılan en yüksek TPR."""
    if len(set(y)) < 2:
        return float("nan")
    fpr, tpr, _ = roc_curve(y, p)
    return float(np.max(tpr[fpr <= hedef_fpr]))


def olc(y, p, esik_sabit):
    y = np.asarray(y); p = np.asarray(p)
    tah = (p >= 0.5).astype(int)                    # varsayılan eşik 0,5
    tn = int(((tah == 0) & (y == 0)).sum()); fp = int(((tah == 1) & (y == 0)).sum())
    fn = int(((tah == 0) & (y == 1)).sum()); tp = int(((tah == 1) & (y == 1)).sum())
    iki_sinif = len(set(y)) == 2
    tah_s = (p > esik_sabit).astype(int)            # eğitim tarafında seçilmiş sabit eşik
    masum_say = int((y == 0).sum()); zararli_say = int((y == 1).sum())
    return {
        "n": len(y), "n_zararli": zararli_say, "n_masum": masum_say,
        "dogruluk": accuracy_score(y, tah),
        "kesinlik": precision_score(y, tah, zero_division=0),
        "duyarlilik": recall_score(y, tah, zero_division=0),
        "f1": f1_score(y, tah, zero_division=0),
        "roc_auc": roc_auc_score(y, p) if iki_sinif else float("nan"),
        "TP": tp, "FN": fn, "FP": fp, "TN": tn,
        "tpr_roc_1fpr_(a)": tpr_roc(y, p),
        "esik_sabit": esik_sabit,
        "tpr_sabit_(b)": float(((tah_s == 1) & (y == 1)).sum() / max(zararli_say, 1)),
        "fpr_sabit_(b)": float(((tah_s == 1) & (y == 0)).sum() / max(masum_say, 1)),
    }

## 7) Model yükleme, eğitim ve tahmin fonksiyonları
- `tokenizer` = metni sayılara çeviren "kelime parçalayıcı".
- Eğitim, 02'deki gibi: her turda tüm örnekleri 16'lık partilerle gezer, hatayı (loss) hesaplar, modeli düzeltir.
- GPU'da yarı hassasiyet (fp16) kullanılır, 02'den tek fark budur. Sadece hız içindir.

In [ ]:
def modeli_yukle(model_adi):
    tokenizer = AutoTokenizer.from_pretrained(model_adi)
    model = AutoModelForSequenceClassification.from_pretrained(model_adi, num_labels=2)
    return tokenizer, model.to(cihaz)


def kodla(tokenizer, metinler):
    k = tokenizer(list(metinler), padding="max_length", truncation=True,
                  max_length=MAKS_UZUNLUK, return_tensors="pt")
    return k["input_ids"], k["attention_mask"]


def egit(model, tokenizer, egitim_df):
    ids, maske = kodla(tokenizer, egitim_df["text"])
    etiket = torch.tensor(egitim_df["label"].astype(int).values)
    uretec = torch.Generator().manual_seed(TOHUM)
    yukleyici = DataLoader(TensorDataset(ids, maske, etiket), batch_size=PARTI,
                           shuffle=True, generator=uretec)
    optimizer = torch.optim.AdamW(model.parameters(), lr=OGRENME_HIZI)
    olcekleyici = torch.amp.GradScaler("cuda", enabled=KARISIK)
    for tur in range(TUR):
        model.train(); toplam = 0.0
        for i, m, e in yukleyici:
            i, m, e = i.to(cihaz), m.to(cihaz), e.to(cihaz)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=KARISIK):
                cikti = model(input_ids=i, attention_mask=m, labels=e)   # tahmin + hata
            optimizer.zero_grad()
            olcekleyici.scale(cikti.loss).backward()   # hatanın nereden geldiğini hesapla
            olcekleyici.step(optimizer)                # modeli düzelt
            olcekleyici.update()
            toplam += cikti.loss.item()
        print(f"  Tur {tur+1}/{TUR} | ortalama loss: {toplam/len(yukleyici):.4f}")


def olasilik_tahmin(model, tokenizer, metinler, parti=64):
    """Her metin için 'zararlı olma olasılığını' (0-1) döndürür. Eğitim yapmaz."""
    model.eval()
    ids, maske = kodla(tokenizer, metinler)
    sonuc = []
    with torch.no_grad():
        for b in range(0, len(ids), parti):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=KARISIK):
                mantik = model(input_ids=ids[b:b+parti].to(cihaz),
                               attention_mask=maske[b:b+parti].to(cihaz)).logits
            sonuc.append(torch.softmax(mantik.float(), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(sonuc)

## 8) Bir koşuyu baştan sona yapan fonksiyon
Bir koşu = (model, çıkarılan veri seti). Adımlar: veriyi böl → modeli yükle → eğit → eşik seç → her test setinde ölç → sonuçları CSV'ye ekle.

In [ ]:
def kosu(model_adi, kaynak_cikarilan):
    ad = f"{model_adi} | cikarilan={kaynak_cikarilan or 'YOK (ayni dagilim referansi)'}"
    print("=" * 70); print("KOŞU:", ad)
    t0 = time.time()
    tohum_ayarla()
    egitim, esik_seti, testler = kosu_verisi(df, kaynak_cikarilan)
    print(f"Eğitim: {len(egitim)} (zararlı oranı {egitim['label'].mean():.3f}) | "
          f"Eşik seti: {len(esik_seti)} (masum: {(esik_seti['label']==0).sum()})")
    if (esik_seti["label"] == 0).sum() < 200:
        print("  UYARI: eşik setinde 200'den az masum var, %1 FPR eşiği kaba bir tahmindir.")

    tokenizer, model = modeli_yukle(model_adi)
    egit(model, tokenizer, egitim)

    p_esik = olasilik_tahmin(model, tokenizer, esik_seti["text"])
    esik = esik_sec(esik_seti["label"].values, p_esik)
    print(f"Eşik (eşik setinde masumların %{HEDEF_FPR*100:.0f}'ini engelleyen): {esik:.6f}")

    satirlar, hatalar = [], []
    for test_adi, t in testler.items():
        p = olasilik_tahmin(model, tokenizer, t["text"])
        r = olc(t["label"].values, p, esik)
        r.update({"model": model_adi, "cikarilan": kaynak_cikarilan or "YOK", "test": test_adi})
        satirlar.append(r)
        yanlis = np.where((p >= 0.5).astype(int) != t["label"].values)[0]
        for j in yanlis:   # modelin yanıldığı örnekler (ilk 300 karakter)
            hatalar.append({"model": model_adi, "cikarilan": kaynak_cikarilan or "YOK", "test": test_adi,
                            "gercek_etiket": int(t["label"].values[j]), "olasilik": float(p[j]),
                            "kategori": t["kategori"].values[j], "text": t["text"].values[j][:300]})
    sure = (time.time() - t0) / 60
    tablo = pd.DataFrame(satirlar)
    tablo["sure_dk"] = round(sure, 1)
    sutunlar = ["model", "cikarilan", "test", "n", "n_zararli", "n_masum", "dogruluk", "kesinlik",
                "duyarlilik", "f1", "roc_auc", "TP", "FN", "FP", "TN", "tpr_roc_1fpr_(a)",
                "esik_sabit", "tpr_sabit_(b)", "fpr_sabit_(b)", "sure_dk"]
    tablo = tablo[sutunlar]

    # Her koşudan sonra Drive'a ekle (Colab kopsa bile kaybolmasın)
    tablo.to_csv(SONUC_DOSYASI, mode="a", header=not os.path.exists(SONUC_DOSYASI), index=False)
    if hatalar:
        pd.DataFrame(hatalar).to_csv(HATA_DOSYASI, mode="a", header=not os.path.exists(HATA_DOSYASI), index=False)

    # GPU belleğini boşalt (sonraki koşu için)
    del model, tokenizer
    if cihaz == "cuda":
        torch.cuda.empty_cache()
    print(f"Koşu süresi: {sure:.1f} dk")
    return tablo

## 9) TEK KOŞU: boru hattını doğrula (safeguard'ı dışarıda bırak, BERT)
**Önce sadece bunu çalıştır.** Birkaç dakika sürmesi beklenir (GPU'da). Bittiğinde çıktıyı (özellikle en alttaki tablo ve `Eşik` satırı) Claude'a yapıştır.

In [ ]:
sonuc = kosu("bert-base-uncased", "safeguard")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
sonuc.round(4)

## 10) TÜM KOŞULAR (tek koşu düzgün çıktıktan SONRA çalıştır)
8 koşu: 2 model x (3 leave-one-dataset-out + 1 aynı dağılım referansı). Daha önce bitmiş koşuları atlar, yani Colab kopsa bile yeniden çalıştırınca kaldığı yerden devam eder.

Not: Şimdilik **kategori/aile genelleme deneyi** (isteğe bağlı) bu defterde yok, sonra eklenecek.

In [ ]:
MODELLER = ["bert-base-uncased", "roberta-base"]
CIKARILANLAR = ["safeguard", "neuralchemy", "wambosec", None]   # None = aynı dağılım referansı

def bitti_mi(model_adi, cikarilan):
    if not os.path.exists(SONUC_DOSYASI):
        return False
    s = pd.read_csv(SONUC_DOSYASI)
    return ((s["model"] == model_adi) & (s["cikarilan"] == (cikarilan or "YOK"))).any()

for model_adi in MODELLER:
    for cikarilan in CIKARILANLAR:
        if bitti_mi(model_adi, cikarilan):
            print("ATLANDI (zaten bitmiş):", model_adi, cikarilan)
            continue
        kosu(model_adi, cikarilan)

tum = pd.read_csv(SONUC_DOSYASI)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
tum.round(4)

11

In [ ]:
import pandas as pd
h = pd.read_csv(HATA_DOSYASI)   # modelin yanıldığı örneklerin kayıtlı olduğu dosya
# safeguard testinde, aslında MASUM (gercek_etiket=0) olup zararlı sanılan örnekler
yanlis_alarm = h[(h["test"] == "safeguard (tamami)") & (h["gercek_etiket"] == 0)]
print("Yanlış alarm sayısı:", len(yanlis_alarm))
for m in yanlis_alarm.sample(8, random_state=1)["text"]:   # rastgele 8 örnek
    print("-", m[:200])

In [ ]:
# Her ana veri setinden 6 rastgele MASUM (label=0) örnek ve ortalama uzunluk
for k in ["neuralchemy", "wambosec", "safeguard"]:
    masum = df[(df["kaynak"] == k) & (df["label"] == 0)]          # sadece masum satırlar
    print(f"=== {k} | masum sayısı: {len(masum)} | ortalama uzunluk: {masum['text'].str.len().mean():.0f} karakter")
    for m in masum.sample(6, random_state=1)["text"]:             # rastgele 6 örnek
        print("  -", m[:150].replace("\n", " "))

In [ ]:
# Tekrarlı koşu satırlarını silip YENİ dosyalara yaz (eskiler kalır)
s = pd.read_csv(SONUC_DOSYASI).drop_duplicates()
h = pd.read_csv(HATA_DOSYASI).drop_duplicates()
s.to_csv(SONUC_KLASORU + "/sonuclar_ahmet_temiz.csv", index=False)
h.to_csv(HATA_KLASORU + "/hatalar_ahmet_temiz.csv", index=False) if False else h.to_csv(SONUC_KLASORU + "/hatalar_ahmet_temiz.csv", index=False)
print("sonuç satırı:", len(s), "| hata satırı:", len(h))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

def ozellikler(metinler):
    # Sadece metnin yüzeysel özellikleri: uzunluk, kelime sayısı, satır sayısı, ':' sayısı
    s = pd.Series(list(metinler))
    return np.column_stack([np.log1p(s.str.len()), np.log1p(s.str.split().str.len()),
                            s.str.count("\\n"), s.str.count(":")])

ana = df[df["rol"] == "ana"]
egit = ana[(ana["kaynak"] != "safeguard") & ana["bolum"].isin(["train", "val"])]   # neuralchemy + wambosec
test = ana[ana["kaynak"] == "safeguard"]                                           # eğitimde yok

basit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
basit.fit(ozellikler(egit["text"]), egit["label"])
p = basit.predict_proba(ozellikler(test["text"]))[:, 1]
print("Sadece uzunluk/biçim özellikleriyle safeguard ROC-AUC:", round(roc_auc_score(test["label"], p), 3))

# Zararlı ve masum örneklerin ortanca uzunluğu (kaynak bazında)
print(df[df["rol"] == "ana"].groupby(["kaynak", "label"])["text"].apply(lambda s: s.str.len().median()))